In [1]:
import pandas as pd # Importa la librería para manipulación y análisis de datos en DataFrames
import numpy as np # Importa la librería para cálculos numéricos avanzados y manejo de arreglos
import os # Importa el módulo para interactuar con el sistema de archivos (crear carpetas, rutas)
import gc # Importa el recolector de basura (Garbage Collector) para liberar memoria RAM manualmente
import time # Importa el módulo para medir el tiempo de ejecución de los procesos
from sklearn.model_selection import GridSearchCV # Importa la herramienta para la búsqueda exhaustiva de hiperparámetros
from sklearn.model_selection import StratifiedKFold # Importa la herramienta para validación cruzada estratificada (mantiene proporción de clases)
from sklearn.ensemble import RandomForestClassifier # Importa el algoritmo base de aprendizaje automático Random Forest
from sklearn.base import clone # Importa la función para clonar la arquitectura de un modelo sin sus datos entrenados
from sklearn.preprocessing import label_binarize # Importa la función para convertir etiquetas multiclase en formato binario (One-vs-Rest)
from sklearn.metrics import ( # Importa el bloque de métricas de evaluación
    f1_score, # Métrica que evalúa el balance entre precisión y exhaustividad (recall)
    average_precision_score, # Métrica para calcular el Área Bajo la Curva Precision-Recall (AUPRC)
    roc_auc_score, # Métrica para calcular el Área Bajo la Curva ROC (AUC-ROC)
    brier_score_loss, # Métrica para evaluar la calibración de las probabilidades predichas
    classification_report # Función que genera un reporte de texto con precision, recall y f1 por clase
)

def entrenar_evaluar_rf(target_name):
    """
    - Descripción: Función que entrena, optimiza y evalúa un modelo Random Forest en base a un target (etiqueta) específico. Aplica validación cruzada 
                   estratificada con semilla fija, descarta hiperparámetros inestables (con desviación estándar > 0.10) y calcula métricas diferenciadas
                   dependiendo si el problema es binario (Mortalidad) o multiclase (Severidad, Consumo de Recursos).
    - Entrada: target_name (str) - Nombre de la columna objetivo a predecir (ej. 'MORTALIDAD', 'SEVERIDAD', 'CONSUMO_RECURSOS').
    - Salida: Genera archivos CSV con el historial del GridSearch y las importancias de variables.
    """
    # 1. Configuración inicial
    dir_datos = "../../Datos/Datasets Finales" # Ruta donde están los CSV de entrada
    dir_resultados = "../../Resultados/Resultados (etapa 3)/Random_Forest" # Define la ruta donde se guardarán los resultados
    os.makedirs(dir_resultados, exist_ok=True) # Crea la carpeta de resultados si es que aún no existe en el sistema

    cols_excluir = ['CONSUMO_RECURSOS', 'SEVERIDAD', 'MORTALIDAD', 'CATEGORIA_CANCER'] # Lista de columnas a excluir (que no se usarán como predictores)

    print("="*60) # Separador visual en la consola
    print(f"INICIANDO PIPELINE DE RANDOM FOREST PARA: {target_name.upper()}") # Imprime el nombre de la variable objetivo actual
    print("="*60) # Separador visual en la consola

    # 2. Cargar datos de entrenamiento
    print("[1/5] Cargando datasets de entrenamiento...") # Etapa 1: Carga de datos de entrenamiento
    df_onco_train = pd.read_csv(os.path.join(dir_datos, "dataset_entrenamiento_onco_2019_2023.csv"), low_memory=False) # Carga los pacientes con cáncer (entrenamiento)
    df_control_train = pd.read_csv(os.path.join(dir_datos, "dataset_entrenamiento_control_2019_2023.csv"), low_memory=False) # Carga los pacientes de control (sin cáncer) (entrenamiento)

    # 3. Crear Dataset Maestro Balanceado (Oncológico vs Control)
    print("[2/5] Generando muestra balanceada...") # Etapa 2: Balanceo de clases para entrenamiento
    n_onco = len(df_onco_train) # Obtiene la cantidad exacta de pacientes oncológicos
    df_control_sample = df_control_train.sample(n=n_onco, random_state=42) # Extrae una muestra aleatoria de control del mismo tamaño, fijando la semilla en 42
    df_train_maestro = pd.concat([df_onco_train, df_control_sample], ignore_index=True) # Une ambos DataFrames apilándolos verticalmente (control y oncológico)

    del df_onco_train, df_control_train, df_control_sample # Elimina los DataFrames originales para liberar RAM
    gc.collect() # Fuerza al recolector de basura a limpiar la memoria inmediatamente

    # Separar X (predictores) e Y (objetivo o targets) de entrenamiento
    features = [col for col in df_train_maestro.columns if col not in cols_excluir] # Crea una lista con los nombres de las columnas que sí se usarán para predecir
    X_train = df_train_maestro[features] # Crea la matriz de predictores usando solo las columnas permitidas
    y_train = df_train_maestro[target_name] # Aísla la columna objetivo (la respuesta correcta: etiqueta)
    
    clases_unicas = np.unique(y_train) # Extrae un arreglo con las clases únicas presentes en la variable objetivo (ej. [0, 1])
    # Determina lógicamente (True/False) si el problema tiene más de 2 clases 
    is_multiclass = len(clases_unicas) > 2 # Por ejemplo, mortalidad tiene 2 clases, severidad tiene 4 clases, consumo de recursos tiene 3 clases
    # Imprime las dimensiones y el tipo de problema (binario o multiclase) 
    print(f"      -> Dimensiones: {X_train.shape} | Clases: {len(clases_unicas)} (Multiclase: {is_multiclass})") 

    # 4. Configurar Grid Search (5 Pliegues) con semilla fija para reproducibilidad del entrenamiento
    print("[3/5] Configurando Grid Search CV (K=5) ...") # Etapa 3: Configuración de la búsqueda de hiperparámetros con validación cruzada
    
    # Configura la validación cruzada estratificada con cortes aleatorios fijos (semilla 42)
    cv_estrategia = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    rf_base = RandomForestClassifier( # Inicializa el modelo base Random Forest
        class_weight='balanced', # Ajusta los pesos internamente para combatir el desbalance de clases
        random_state=42, # Fija la semilla matemática de los árboles para reproducibilidad
        n_jobs=-1 # Habilita el paralelismo usando todos los hilos del procesador
    )

    espacio_hiperparametros = { # Define el diccionario con la grilla de búsqueda según la propuesta metodológica
        'n_estimators': [100, 200, 500], # Valores a probar para la cantidad de árboles
        'max_depth': [10, 20, None], # Valores a probar para la profundidad máxima
        'min_samples_split': [2, 5, 10] # Valores a probar para la mínima cantidad de muestras requeridas para dividir un nodo
    }

    grid_search = GridSearchCV( # Inicializa el orquestador de búsqueda de hiperparámetros
        estimator=rf_base, # Le indica qué modelo entrenar
        param_grid=espacio_hiperparametros, # Le entrega la grilla de configuraciones a probar
        cv=cv_estrategia, # Le entrega la estrategia de 5 pliegues con semilla fija
        scoring='f1_macro', # Le indica que la métrica a maximizar es el F1-Score Promedio
        n_jobs=1, # Evalúa de a un pliegue a la vez para no colapsar la RAM
        verbose=3 # Nivel de detalle aumentado: Imprime cada pliegue, puntaje y tiempo exacto tomado en consola
    )

    # 5. Entrenar y evaluar configuraciones
    print("[4/5] Entrenando modelo y evaluando configuraciones ...") # Etapa 4: Entrenamiento y evaluación de cada configuración de hiperparámetros
    inicio = time.time() # Registra el tiempo exacto de inicio
    grid_search.fit(X_train, y_train) # Ejecuta los 135 entrenamientos (27 configuraciones x 5 pliegues)
    fin = time.time() # Registra el tiempo exacto de finalización
    print(f"      -> Búsqueda completada en {round((fin - inicio)/60, 2)} minutos.") # Imprime la duración total en minutos

    # -------------------------------------------------------------------------
    # Extracción de resultados, generación de respaldo csv y filtrado metodológico.
    # -------------------------------------------------------------------------
    cv_resultados = pd.DataFrame(grid_search.cv_results_) # Convierte el diccionario interno de resultados en un DataFrame de Pandas
    # Construye la ruta para guardar el historial de resultados del GridSearch específico para el target actual
    ruta_cv = os.path.join(dir_resultados, f"Resultados_GridSearch_RF_{target_name}.csv") 
    cv_resultados.to_csv(ruta_cv, index=False) # Exporta el DataFrame a formato CSV sin la columna de índices
    print(f"      -> Evidencia de hiperparámetros guardada en: {ruta_cv}") # Confirma la creación del csv de respaldo
    # Filtra dejando solo las configuraciones con desviación estándar menor o igual a 0.10, es decir, las más estables
    config_estables = cv_resultados[cv_resultados['std_test_score'] <= 0.10] 

    if config_estables.empty: # Verifica si el DataFrame filtrado quedó vacío (es decir, todas fueron inestables)
        print("      ADVERTENCIA: Todas las configuraciones tienen desviación estándar (std) > 0.10.") # Advierte sobre la inestabilidad total
        print("      Se utilizará la de mayor promedio por defecto de Scikit-Learn.") # Indica el plan de contingencia
        mejor_modelo = grid_search.best_estimator_ # Adopta el mejor modelo que scikit-learn calculó originalmente por defecto
    else: # Si existen configuraciones estables (lo ideal)
        mejor_indice = config_estables['mean_test_score'].idxmax() # Encuentra el índice (fila) que tiene el valor máximo en la columna de F1 promedio
        mejores_params = config_estables.loc[mejor_indice, 'params'] # Extrae el diccionario exacto de hiperparámetros de esa fila ganadora
        mejor_f1 = config_estables.loc[mejor_indice, 'mean_test_score'] # Extrae el valor numérico del mejor F1 promedio
        mejor_std = config_estables.loc[mejor_indice, 'std_test_score'] # Extrae el valor numérico de su desviación estándar
        
        print(f"      -> Mejor configuración estable encontrada:") # Confirma el hallazgo
        print(f"         Hiperparámetros: {mejores_params}") # Imprime la configuración ganadora
        print(f"         F1-Macro Promedio: {mejor_f1:.4f} (std: {mejor_std:.4f})") # Imprime sus métricas asociadas

        mejor_modelo = clone(grid_search.estimator) # Clona la arquitectura de RF vacía (con random_state y n_jobs, pero sin datos)
        mejor_modelo.set_params(**mejores_params) # Le inyecta los hiperparámetros ganadores extraídos del filtro
        mejor_modelo.fit(X_train, y_train) # Entrena este modelo definitivo usando el 100% de los datos de la matriz de entrenamiento
        
    # -------------------------------------------------------------------------
    del df_train_maestro, X_train, y_train # Elimina los objetos pesados de la fase de entrenamiento
    gc.collect() # Limpia la RAM nuevamente

    # 6. Evaluación en el Conjunto de Prueba
    print("[5/5] Evaluando en conjunto de prueba (2024)...") # Etapa 5: Evaluación del modelo final en el conjunto de prueba de 2024
    df_onco_test = pd.read_csv(os.path.join(dir_datos, "dataset_prueba_onco_2024.csv"), low_memory=False) # Carga los pacientes con cáncer de 2024
    df_control_test = pd.read_csv(os.path.join(dir_datos, "dataset_prueba_control_2024.csv"), low_memory=False) # Carga los controles de 2024

    df_test_maestro = pd.concat([df_onco_test, df_control_test], ignore_index=True) # Une ambas muestras de prueba verticalmente
    X_test = df_test_maestro[features] # Aísla las variables predictoras del conjunto de prueba
    y_test = df_test_maestro[target_name] # Aísla la variable objetivo (respuestas correctas) de 2024

    y_pred = mejor_modelo.predict(X_test) # Pide al modelo que prediga las clases (ej. 0 o 1) para el conjunto de prueba
    y_pred_proba = mejor_modelo.predict_proba(X_test) # Pide al modelo las probabilidades continuas asociadas a esas predicciones (ej. 85% de morir)

    print("\n--- RESULTADOS FINALES ---") 
    print(classification_report(y_test, y_pred)) # Imprime el reporte estándar con precision, recall y soporte por clase
    
    f1_macro_val = f1_score(y_test, y_pred, average='macro') # Calcula la métrica F1-Macro general
    
    if is_multiclass: # Si el problema evaluado es de Severidad o Consumo de Recursos
        y_test_bin = label_binarize(y_test, classes=clases_unicas) # Convierte las clases reales en matriz binaria (One-vs-Rest)
        auc_roc_val = roc_auc_score(y_test, y_pred_proba, multi_class='ovr', average='weighted') # Calcula AUC-ROC con estrategia One-vs-Rest ponderada por soporte
        auprc_val = average_precision_score(y_test_bin, y_pred_proba, average='weighted') # Calcula AUPRC multiclase ponderada
        # Calcula el promedio del Brier Score calculado individualmente por clase
        brier_val = np.mean([brier_score_loss(y_test_bin[:, k], y_pred_proba[:, k]) for k in range(len(clases_unicas))]) 
        print(f"F1-Score (Macro): {f1_macro_val:.4f}") # Muestra F1 en pantalla
        print(f"AUPRC (OvR Weighted): {auprc_val:.4f}") # Muestra AUPRC en pantalla
        print(f"AUC-ROC (OvR Weighted): {auc_roc_val:.4f}") # Muestra AUC-ROC en pantalla
        print(f"Brier Score (Multiclase): {brier_val:.4f}") # Muestra Brier Score en pantalla
            
    else: # Si el problema evaluado es de Mortalidad (Binario)
        f1_clase1_val = f1_score(y_test, y_pred, pos_label=1) # Calcula el F1 enfocado solo en la clase minoritaria (fallecidos)
        auc_roc_val = roc_auc_score(y_test, y_pred_proba[:, 1]) # Calcula el AUC-ROC usando la probabilidad de la clase positiva
        auprc_val = average_precision_score(y_test, y_pred_proba[:, 1]) # Calcula el AUPRC usando la probabilidad de la clase positiva
        brier_val = brier_score_loss(y_test, y_pred_proba[:, 1]) # Calcula el Brier Score evaluando qué tan reales son las probabilidades de la clase 1
        
        print(f"F1-Score (Clase 1): {f1_clase1_val:.4f}") # Muestra F1 minoritario
        print(f"F1-Score (Macro): {f1_macro_val:.4f}") # Muestra F1 general
        print(f"AUPRC: {auprc_val:.4f}") # Muestra AUPRC
        print(f"AUC-ROC: {auc_roc_val:.4f}") # Muestra AUC-ROC
        print(f"Brier Score: {brier_val:.4f}") # Muestra calibración (Brier)

    # 7. Extraer Feature Importances
    importancias = mejor_modelo.feature_importances_ # Extrae el arreglo de pesos relativos (importancias) que el árbol le dio a cada variable
    
    df_importancias = pd.DataFrame({ # Crea un DataFrame nuevo para ordenar las importancias
        'Variable': features, # Columna con el nombre de los predictores
        'Importancia_Relativa': importancias # Columna con el valor numérico de la importancia
    }).sort_values(by='Importancia_Relativa', ascending=False) # Ordena la tabla de mayor a menor importancia
    
    df_importancias = df_importancias[df_importancias['Importancia_Relativa'] > 0] # Limpia el reporte descartando atributos con 0 aporte
    
    ruta_imp = os.path.join(dir_resultados, f"RF_Feature_Importances_{target_name}.csv") # Construye la ruta de guardado del CSV
    df_importancias.to_csv(ruta_imp, index=False) # Exporta la tabla sin la columna de índice
    
    print(f"Importancias de Variables guardadas en: {ruta_imp}") # Confirma la creación del archivo
    
    del df_test_maestro, X_test, y_test # Destruye las variables de prueba
    gc.collect() # Hace la limpieza final de memoria RAM
    print("="*60, "\n")

In [2]:
entrenar_evaluar_rf('MORTALIDAD')

INICIANDO PIPELINE DE RANDOM FOREST PARA: MORTALIDAD
[1/5] Cargando datasets de entrenamiento...
[2/5] Generando muestra balanceada...
      -> Dimensiones: (781264, 110) | Clases: 2 (Multiclase: False)
[3/5] Configurando Grid Search CV (K=5) ...
[4/5] Entrenando modelo y evaluando configuraciones ...
Fitting 5 folds for each of 27 candidates, totalling 135 fits
[CV 1/5] END max_depth=10, min_samples_split=2, n_estimators=100;, score=0.567 total time=  33.9s
[CV 2/5] END max_depth=10, min_samples_split=2, n_estimators=100;, score=0.568 total time=  34.2s
[CV 3/5] END max_depth=10, min_samples_split=2, n_estimators=100;, score=0.568 total time=  34.0s
[CV 4/5] END max_depth=10, min_samples_split=2, n_estimators=100;, score=0.568 total time=  34.3s
[CV 5/5] END max_depth=10, min_samples_split=2, n_estimators=100;, score=0.569 total time=  33.8s
[CV 1/5] END max_depth=10, min_samples_split=2, n_estimators=200;, score=0.569 total time= 1.1min
[CV 2/5] END max_depth=10, min_samples_split=2,

In [3]:
entrenar_evaluar_rf('SEVERIDAD')

INICIANDO PIPELINE DE RANDOM FOREST PARA: SEVERIDAD
[1/5] Cargando datasets de entrenamiento...
[2/5] Generando muestra balanceada...
      -> Dimensiones: (781264, 110) | Clases: 4 (Multiclase: True)
[3/5] Configurando Grid Search CV (K=5) ...
[4/5] Entrenando modelo y evaluando configuraciones ...
Fitting 5 folds for each of 27 candidates, totalling 135 fits
[CV 1/5] END max_depth=10, min_samples_split=2, n_estimators=100;, score=0.693 total time=  35.8s
[CV 2/5] END max_depth=10, min_samples_split=2, n_estimators=100;, score=0.692 total time=  35.7s
[CV 3/5] END max_depth=10, min_samples_split=2, n_estimators=100;, score=0.694 total time=  35.7s
[CV 4/5] END max_depth=10, min_samples_split=2, n_estimators=100;, score=0.690 total time=  35.5s
[CV 5/5] END max_depth=10, min_samples_split=2, n_estimators=100;, score=0.692 total time=  35.7s
[CV 1/5] END max_depth=10, min_samples_split=2, n_estimators=200;, score=0.696 total time= 1.2min
[CV 2/5] END max_depth=10, min_samples_split=2, n

In [4]:
entrenar_evaluar_rf('CONSUMO_RECURSOS')

INICIANDO PIPELINE DE RANDOM FOREST PARA: CONSUMO_RECURSOS
[1/5] Cargando datasets de entrenamiento...
[2/5] Generando muestra balanceada...
      -> Dimensiones: (781264, 110) | Clases: 3 (Multiclase: True)
[3/5] Configurando Grid Search CV (K=5) ...
[4/5] Entrenando modelo y evaluando configuraciones ...
Fitting 5 folds for each of 27 candidates, totalling 135 fits
[CV 1/5] END max_depth=10, min_samples_split=2, n_estimators=100;, score=0.686 total time=  32.1s
[CV 2/5] END max_depth=10, min_samples_split=2, n_estimators=100;, score=0.688 total time=  31.9s
[CV 3/5] END max_depth=10, min_samples_split=2, n_estimators=100;, score=0.683 total time=  31.9s
[CV 4/5] END max_depth=10, min_samples_split=2, n_estimators=100;, score=0.687 total time=  32.0s
[CV 5/5] END max_depth=10, min_samples_split=2, n_estimators=100;, score=0.685 total time=  32.1s
[CV 1/5] END max_depth=10, min_samples_split=2, n_estimators=200;, score=0.686 total time= 1.1min
[CV 2/5] END max_depth=10, min_samples_spl

In [2]:
import pandas as pd # Importa Pandas para leer los archivos CSV
import numpy as np # Importa Numpy para cálculos matemáticos
import os # Importa OS para manejar las rutas de los archivos

def validar_auprc_lift_automatico(target_name, auprc_obtenido, is_multiclass=False):
    """
    - Descripción: Lee automáticamente el conjunto de prueba (2024), calcula la tasa base (prevalencia)
                   directamente de los datos y verifica matemáticamente si el AUPRC cumple con Lift > 3.0.
    - Entrada: target_name (str) - Nombre del objetivo, auprc_obtenido (float) - El AUPRC de tu mejor modelo,
               is_multiclass (bool) - True si tiene más de 2 clases.
    - Salida: Imprime el reporte de validación en la consola.
    """
    print("="*60) # Separador visual
    print(f"VALIDACIÓN AUTOMÁTICA DE AUPRC: {target_name.upper()}") # Título
    print("="*60) # Separador visual
    
    # 1. Leer SOLO la columna objetivo de los archivos de 2024 (es ultrarrápido y no gasta RAM)
    dir_datos = "../../Datos/Datasets Finales" # Define la ruta relativa
    df_onco = pd.read_csv(os.path.join(dir_datos, "dataset_prueba_onco_2024.csv"), usecols=[target_name]) # Carga onco
    df_control = pd.read_csv(os.path.join(dir_datos, "dataset_prueba_control_2024.csv"), usecols=[target_name]) # Carga control
    
    # 2. Unir y contar las instancias reales
    y_test = pd.concat([df_onco, df_control], ignore_index=True)[target_name] # Une ambos vectores
    clases, soportes_clases = np.unique(y_test, return_counts=True) # Extrae las clases y cuenta cuántas hay de cada una
    total_instancias = len(y_test) # Obtiene el total de pacientes evaluados
    
    # 3. Calcular la Tasa Base según el tipo de problema
    if not is_multiclass: # Si es Mortalidad (Binario)
        indice_clase_1 = np.where(clases == 1)[0][0] # Encuentra en qué posición del arreglo está la clase 1 (Fallecidos)
        soporte_clase_positiva = soportes_clases[indice_clase_1] # Extrae la cantidad exacta de fallecidos
        tasa_base = soporte_clase_positiva / total_instancias # Calcula la prevalencia pura
        
        print(f"Total episodios de prueba: {total_instancias}") # Muestra total
        print(f"Casos reales clase minoritaria (1): {soporte_clase_positiva}") # Muestra casos positivos
        print(f"Tasa base (Prevalencia): {tasa_base:.4f} ({tasa_base*100:.2f}%)") # Muestra porcentaje
        
    else: # Si es Severidad o Consumo (Multiclase)
        prevalencias = [soporte / total_instancias for soporte in soportes_clases] # Calcula la prevalencia individual de cada clase
        tasa_base = sum([p**2 for p in prevalencias]) # Calcula la prevalencia esperada por azar (promedio ponderado OvR)
        
        print(f"Total episodios de prueba: {total_instancias}") # Muestra total
        print(f"Prevalencias exactas por clase: {[round(p, 4) for p in prevalencias]}") # Muestra cómo se reparte la torta
        print(f"Tasa base ponderada (OvR): {tasa_base:.4f} ({tasa_base*100:.2f}%)") # Muestra el azar ponderado

    # 4. Calcular el umbral y el Lift real
    umbral_minimo = tasa_base * 3.0 # Calcula el AUPRC mínimo que exige tu tesis
    lift_real = auprc_obtenido / tasa_base # Calcula cuántas veces mejor que el azar es tu modelo
    
    print("-" * 60) # Separador
    print(f"- AUPRC Obtenido por tu modelo: {auprc_obtenido:.4f}") # Muestra tu resultado
    print(f"- AUPRC Mínimo exigido (Tasa Base x 3.0): {umbral_minimo:.4f}") # Muestra el límite inferior
    print(f"- LIFT REAL LOGRADO: {lift_real:.2f}x") # Muestra el multiplicador de mejora
    
    # 5. Veredicto final
    if auprc_obtenido > umbral_minimo: # Si supera el umbral
        print("RESULTADO: ¡CUMPLE LA CONDICIÓN ESTRICTA DE LA TESIS! (Lift > 3.0)")
    else: # Si no lo supera
        print("RESULTADO: NO CUMPLE LA CONDICIÓN (Revisar factibilidad matemática por balanceo de clases)")
    print("\n")

In [3]:
validar_auprc_lift_automatico(
    target_name='MORTALIDAD', 
    auprc_obtenido=0.3561, 
    is_multiclass=False
)

VALIDACIÓN AUTOMÁTICA DE AUPRC: MORTALIDAD
Total episodios de prueba: 1076345
Casos reales clase minoritaria (1): 26221
Tasa base (Prevalencia): 0.0244 (2.44%)
------------------------------------------------------------
- AUPRC Obtenido por tu modelo: 0.3561
- AUPRC Mínimo exigido (Tasa Base x 3.0): 0.0731
- LIFT REAL LOGRADO: 14.62x
RESULTADO: ¡CUMPLE LA CONDICIÓN ESTRICTA DE LA TESIS! (Lift > 3.0)




In [4]:
validar_auprc_lift_automatico(
    target_name='SEVERIDAD', 
    auprc_obtenido=0.7694, 
    is_multiclass=True
)

VALIDACIÓN AUTOMÁTICA DE AUPRC: SEVERIDAD
Total episodios de prueba: 1076345
Prevalencias exactas por clase: [0.1969, 0.3561, 0.2478, 0.1992]
Tasa base ponderada (OvR): 0.2667 (26.67%)
------------------------------------------------------------
- AUPRC Obtenido por tu modelo: 0.7694
- AUPRC Mínimo exigido (Tasa Base x 3.0): 0.8000
- LIFT REAL LOGRADO: 2.89x
RESULTADO: NO CUMPLE LA CONDICIÓN (Revisar factibilidad matemática por balanceo de clases)




In [5]:
validar_auprc_lift_automatico(
    target_name='CONSUMO_RECURSOS', 
    auprc_obtenido=0.8155, 
    is_multiclass=True
)

VALIDACIÓN AUTOMÁTICA DE AUPRC: CONSUMO_RECURSOS
Total episodios de prueba: 1076345
Prevalencias exactas por clase: [0.3345, 0.3367, 0.3287]
Tasa base ponderada (OvR): 0.3334 (33.34%)
------------------------------------------------------------
- AUPRC Obtenido por tu modelo: 0.8155
- AUPRC Mínimo exigido (Tasa Base x 3.0): 1.0001
- LIFT REAL LOGRADO: 2.45x
RESULTADO: NO CUMPLE LA CONDICIÓN (Revisar factibilidad matemática por balanceo de clases)


